# Your first 50x speedup

Here is a slow Python function — a hot loop over a NumPy array. Run the next two cells and watch what one decorator does.

We will explain *why* it works in a moment. For now, just run it.

In [ ]:
import numpy as np
import numba
from numba import jit

@jit
def go_fast(a):  # Function is compiled to machine code when called the first time
    trace = 0.0
    for i in range(a.shape[0]):   # Numba likes loops
        trace += np.tanh(a[i, i])  # Numba likes NumPy functions
    return a + trace               # Numba likes NumPy broadcasting

x = np.arange(100).reshape(10, 10)
go_fast(x)  # first call compiles the function

In [ ]:
# Compare: the compiled @jit version vs. the original pure-Python version.
# .py_func is the uncompiled original, so we can benchmark both.
np.testing.assert_array_equal(go_fast(x), go_fast.py_func(x))  # same answer

numba_time = %timeit -o go_fast(x)
python_time = %timeit -o go_fast.py_func(x)
print(f"\nNumba is {python_time.average / numba_time.average:.1f}x faster than pure Python")

## Why did that work?

Numba is a **just-in-time (JIT) compiler**. When you call a `@jit`-decorated function for the first time, Numba translates it into machine code using [LLVM](https://llvm.org/). The compiled code runs anywhere from **2x** (simple NumPy operations) to **100x** (complex Python loops) faster than the original Python.

Key ideas:

- **`@jit`** is a *decorator* — it wraps your function and returns a compiled version.
- The **`nopython=True`** option used to be recommended, but it is now the **default** behavior of `@jit`, so we can omit it. In this mode, the entire function is compiled so the Python interpreter is not called at all. If Numba cannot compile something, it raises an exception telling you which line needs to change. These exceptions usually point to places in the function that need to be modified in order to achieve better-than-Python performance.
- **Compilation is per-signature** — the first call with a new dtype/shape compiles; subsequent calls reuse the compiled version. That is why we call `go_fast(x)` once before benchmarking.

So the rule of thumb: **decorate a hot loop with `@jit`, call it once to compile, then benchmark.**

## Numba vs. vectorized NumPy

The original Python function used explicit loops, which are very fast in Numba and not very fast in Python. Our example function is so simple, we can create an alternate version of `go_fast` using only NumPy array expressions:

In [ ]:
def go_numpy(a):
    return a + np.tanh(np.diagonal(a)).sum()

np.testing.assert_array_equal(go_fast(x), go_numpy(x))
numpy_time = %timeit -o go_numpy(x)
print(f"Numba is {numpy_time.average / numba_time.average:.1f}x faster than NumPy for this loop")

## Your turn

Write a Numba-compiled function that computes the sum of squares of an array using a Python `for` loop (not `np.sum`). Then benchmark it against `np.sum(a**2)`.

```python
@jit
def sum_of_squares(a):
    # your code here
    pass
```

Think first: for a *simple reduction*, do you expect Numba to beat vectorized NumPy? Try it, then check the solution below.

In [ ]:
# Solution
@jit
def sum_of_squares(a):
    total = 0.0
    for x in a:
        total += x * x
    return total

big = np.random.rand(1_000_000)
sum_of_squares(big)  # compile

numba_time = %timeit -o sum_of_squares(big)
numpy_time = %timeit -o np.sum(big**2)
print(f"\nNumba: {numba_time.average*1e6:.1f} us")
print(f"NumPy: {numpy_time.average*1e6:.1f} us")

**Takeaway:** for a simple reduction like this, vectorized NumPy is hard to beat — it calls an optimized C routine with no Python loop overhead. Numba wins when you have a loop that does **not** vectorize cleanly (e.g. the groupby in `3_numba_groupby_pixels.ipynb`), or when you want to **fuse** several array passes into one compiled function to avoid intermediate temporaries.

This is the single most important judgment call when using Numba: *is my bottleneck a vectorizable NumPy expression, or a loop that doesn't vectorize?* Numba accelerates the latter; it rarely beats the former.

## Supported Python Features

Numba works best when used with NumPy arrays, but Numba also supports other data types out of the box:

* `int`, `float`
* `tuple`, `namedtuple`
* `list` (with some restrictions)
* ... and others.  See the [Reference Manual](https://numba.pydata.org/numba-doc/latest/reference/pysupported.html) for more details.

In particular, tuples are useful for returning multiple values from functions:

In [ ]:
@jit
def spherical_to_cartesian(r, theta, phi):
    '''Convert spherical coordinates (physics convention) to cartesian coordinates'''
    sin_theta = np.sin(theta)
    x = r * sin_theta * np.cos(phi)
    y = r * sin_theta * np.sin(phi)
    z = r * np.cos(theta)
    return x, y, z  # return a tuple

@jit
def random_directions(n, r):
    '''Return ``n`` 3-vectors in random directions with radius ``r``'''
    out = np.empty(shape=(n, 3), dtype=np.float64)
    for i in range(n):
        phi = np.random.uniform(0, 2 * np.pi)
        theta = np.arccos(np.random.uniform(-1, 1))
        x, y, z = spherical_to_cartesian(r, theta, phi)  # unpack a tuple
        out[i] = x, y, z
    return out

random_directions(10, 1.0)

When Numba is translating Python to machine code, it uses the [LLVM](https://llvm.org/) library to do most of the optimization and final code generation.  This automatically enables a wide range of optimizations that you don't even have to think about.  If we were to inspect the output of the compiler for the previous random directions example, we would find that:

* The function body for `spherical_to_cartesian()` was inlined directly into the body of the for loop in `random_directions`, eliminating the overhead of making a function call.
* The separate calls to `sin()` and `cos()` were combined into a single, faster call to an internal `sincos()` function.

These kinds of cross-function optimizations are one of the reasons that Numba can sometimes outperform compiled NumPy code.